# Radial derivative

The $-\partial_\sigma$ of the denominator log-product at $\sigma = 1/2$ isolates prime powers.

**Claim evidenced.** Writing
$L_{S,\sigma}(u) = \log \prod_{p\in S} |1 - p^{-\sigma - iu}|^{-2}$, the radial
derivative at the critical exponent is an explicit prime-power sum:
$$-\partial_\sigma L_{S,\sigma}(u)\Big|_{\sigma=1/2}
  = 2 \sum_{p\in S} \sum_{r\ge 1} (\log p)\, p^{-r/2} \cos(u\, r \log p).$$
We plot the truncated series for primes $< 200$ on $u \in [0, 12]$, mark the
prime-power frequencies $r\log p$, and check the formula against a numerical
finite difference of $L_{S,\sigma}(u)$ in $\sigma$.

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sympy import primerange

u = np.linspace(0, 12, 6001)
primes = list(primerange(2, 200))


def radial_series(u: np.ndarray, primes) -> np.ndarray:
    """2 sum_{p, r} (log p) p^{-r/2} cos(u r log p), truncated when the weight < 1e-12."""
    D = np.zeros_like(u)
    for p in primes:
        lp = math.log(p)
        for r in range(1, 40):
            w = (p ** (-r / 2.0)) * lp
            if w < 1e-12:
                break
            D += 2 * w * np.cos(u * r * lp)
    return D


D = radial_series(u, primes)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(u, D, lw=0.8)
for p in (2, 3, 5, 7, 11):
    for r in (1, 2):
        x = r * math.log(p)
        if x <= 12:
            ax.axvline(x, color="r", ls=":", lw=0.6)
ax.set_xlabel("u")
ax.set_ylabel(r"$-\partial_\sigma L_{S,\sigma}(u)\,|_{\sigma=1/2}$")
ax.set_title("Radial derivative concentrates at prime-power logs (dotted: $r\\log p$)")
fig.tight_layout()
plt.show()

<Figure size 1000x400 with 1 Axes>

In [2]:
# Finite-difference check of the analytic series.
def L(sigma: float, u: np.ndarray, primes) -> np.ndarray:
    """L_{S,sigma}(u) = log prod_p |1 - p^{-sigma - iu}|^{-2}."""
    total = np.zeros_like(u)
    for p in primes:
        total -= 2.0 * np.log(np.abs(1.0 - p ** (-sigma - 1j * u)))
    return total


h = 1e-4
fd = -(L(0.5 + h, u, primes) - L(0.5 - h, u, primes)) / (2 * h)
err = float(np.max(np.abs(fd - D)))
print(f"central difference step h = {h}")
print(f"max |finite difference - analytic series| over u in [0, 12]: {err:.3e}")
assert err < 1e-4

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(u, fd - D, lw=0.7)
ax.set_xlabel("u")
ax.set_ylabel("FD $-$ series")
ax.set_title(f"Pointwise discrepancy (max = {err:.2e})")
fig.tight_layout()
plt.show()

central difference step h = 0.0001
max |finite difference - analytic series| over u in [0, 12]: 6.805e-06


<Figure size 1000x300 with 1 Axes>

**Interpretation.** The truncated series shows a dominant spike at $u = 0$ (all
cosines aligned) and clear structure at the prime-power frequencies: local
features line up with the dotted markers at $r\log p$ for small $p$ and $r$,
which is the finite-$S$ shadow of the 'explicit formula' mechanism — the radial
derivative of the log-denominator turns the Euler product into a
von Mangoldt-weighted cosine sum. The independent finite-difference computation
of $-\partial_\sigma L_{S,\sigma}(u)$ at $\sigma = 1/2$ agrees with the analytic
series to within the expected $O(h^2)$ discretization error ($\sim 10^{-6}$ for
$h = 10^{-4}$), confirming the differentiation was carried out correctly. With
only primes $< 200$ the peaks are broad; sharpening them into genuine
distributions on prime-power logs requires the infinite limit, which no finite
plot establishes.